# Microsoft Agent Framework — Azure OpenAI (Responses API)

V tem vzorčnem kode boste uporabili **Microsoft Agent Framework (MAF)** za ustvarjanje preprostega agenta, podprtega z **Azure OpenAI** z uporabo **Responses API**.

> **Opomba o migraciji:** Ta primer je prej uporabljal Semantic Kernel z GitHub modeli. Bil je migriran na Microsoft Agent Framework, GitHub modeli (opravilo, ki se odpoveduje julija 2026) pa so bili zamenjani z Azure OpenAI, ki podpira Responses API. `OpenAIChatClient` v MAF cilja na stabilno točko Azure OpenAI `/openai/v1/` in privzeto uporablja Responses API.

Namen tega vzorca je prikazati korake, ki bodo kasneje uporabljeni v dodatnih vzorčnih kodah pri uvajanju različnih agentnih vzorcev.


In [ ]:
%pip install agent-framework agent-framework-openai azure-identity -q


## Uvoz potrebnih Python paketov


In [ ]:
import os
import random

from dotenv import load_dotenv
from IPython.display import display, HTML

from agent_framework import tool
from agent_framework.openai import OpenAIChatClient
from azure.identity import AzureCliCredential


## Definiranje orodja

V Microsoft Agent Frameworku je **orodje** preprosta Python funkcija, okrašena z `@tool`, ki jo agent lahko pokliče. Spodaj definiramo orodje, ki vrne naključno počitniško destinacijo in se izogne ponavljanju prejšnje.


In [ ]:
# A list of vacation destinations the tool can choose from.
_DESTINATIONS = [
    "Barcelona, Spain",
    "Paris, France",
    "Berlin, Germany",
    "Tokyo, Japan",
    "Sydney, Australia",
    "New York, USA",
    "Cairo, Egypt",
    "Cape Town, South Africa",
    "Rio de Janeiro, Brazil",
    "Bali, Indonesia",
]

# Track the last destination so repeated calls avoid immediate repeats.
_last_destination: str | None = None


@tool(approval_mode="never_require")
def get_random_destination() -> str:
    """Provides a random vacation destination."""
    global _last_destination
    available = _DESTINATIONS.copy()
    if _last_destination and len(available) > 1:
        available.remove(_last_destination)
    destination = random.choice(available)
    _last_destination = destination
    return destination


In [ ]:
load_dotenv()

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# OpenAIChatClient targets Azure OpenAI's v1 endpoint and uses the Responses API.
# Sign in with `az login` first so AzureCliCredential can authenticate.
chat_client = OpenAIChatClient(
    model=deployment,
    azure_endpoint=endpoint,
    credential=AzureCliCredential(),
)


## Ustvarjanje agenta

Tukaj ustvarimo agenta z imenom `TravelAgent`.

V tem primeru uporabljamo zelo osnovna navodila. Prosto spremenite ta navodila, da boste opazovali, kako se vedenje agenta spreminja.


In [ ]:
agent = chat_client.as_agent(
    name="TravelAgent",
    instructions="You are a helpful AI Agent that can help plan vacations for customers at random destinations",
    tools=[get_random_destination],
)


## Zagon agenta

Zdaj lahko zaženemo agenta. Ustvarimo `AgentSession`, da se agent spomni pogovora skozi posamezne korake, nato pošljemo dva `user_inputs`. Prvi prosi za potovanje; drugi pove, da uporabniku ni bila všeč predlagana možnost in prosi za drugo — agent uporablja zgodovino seje in orodje `get_random_destination`, da odgovori.

Te sporočila lahko prilagodite, da opazite, kako agent drugače reagira. Odgovori so **pretakani** znak za znakom.


In [ ]:
user_inputs = [
    "Plan me a day trip.",
    "I don't like that destination. Plan me another vacation.",
]


async def main():
    # A session keeps conversation history across turns.
    session = agent.create_session()

    for user_input in user_inputs:
        html_output = (
            f"<div style='margin-bottom:10px'>"
            f"<div style='font-weight:bold'>User:</div>"
            f"<div style='margin-left:20px'>{user_input}</div></div>"
        )

        full_response: list[str] = []
        # Stream the agent's response token-by-token. The agent will call the
        # get_random_destination tool automatically when it needs a destination.
        async for chunk in agent.run(user_input, session=session, stream=True):
            full_response.append(str(chunk))

        html_output += (
            "<div style='margin-bottom:20px'>"
            f"<div style='font-weight:bold'>TravelAgent:</div>"
            f"<div style='margin-left:20px; white-space:pre-wrap'>{''.join(full_response)}</div></div><hr>"
        )

        display(HTML(html_output))


await main()


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
